# LLaMA-2-7B W/A BFP Baseline

Quantize the LLaMA decoder `nn.Linear` weights and activations with arithmetic BFP fake quantization, run the linear operation, and return FP16 output for the remaining model operations. One run automatically sweeps **BFP8 through BFP4** with group size 16, a shared E5 maximum exponent, and round-to-nearest-even. The FP16 `lm_head` is excluded so the quantized scope matches the BiE-family experiments. Each format reloads the original FP16 checkpoint before quantization.

This notebook uses fake quantization: BFP values are quantized and dequantized onto FP16 tensors before `F.linear`. It measures the numerical effect and PPL degradation, but does not provide packed BFP storage, a native BFP GEMM kernel, or hardware speedup.

In [ ]:
%pip install -q "transformers==5.13.1" "datasets==4.0.0" accelerate sentencepiece tqdm

In [ ]:
import gc
import json
import os
import platform
import time
import zipfile
from dataclasses import asdict, dataclass, replace
from getpass import getpass
from pathlib import Path

import datasets
import torch
import torch.nn as nn
import torch.nn.functional as F
import transformers
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_ID = "meta-llama/Llama-2-7b-hf"
DATASET_ID = "Salesforce/wikitext"
DATASET_CONFIG = "wikitext-2-raw-v1"
SPLIT = "test"
CONTEXT_LENGTH = 2048
STRIDE = 2048
DROP_REMAINDER = True
EVALUATION_PROTOCOL = "non_overlapping_2048_drop_remainder"
BASELINE_PPL = 5.472103118896484
BASELINE_TOKEN_COUNTS = {
    "source_input_tokens": 341_469,
    "used_input_tokens": 339_968,
    "dropped_input_tokens": 1_501,
    "evaluated_blocks": 166,
    "evaluated_tokens": 339_802,
}
MANTISSA_BITS_SWEEP = (7, 6, 5, 4, 3)  # BFP8, BFP7, BFP6, BFP5, BFP4.

@dataclass(frozen=True)
class BFPConfig:
    block_size: int = 16
    shared_exponent_bits: int = 5
    mantissa_bits: int = 7  # Excludes the sign bit: 1S7M.
    rounding: str = "nearest_even"
    weight_chunk_rows: int = 128
    activation_chunk_rows: int = 2048
    quantize_lm_head: bool = False

    def validate(self):
        if self.block_size != 16:
            raise ValueError("This experiment is fixed to G16.")
        if self.shared_exponent_bits != 5:
            raise ValueError("This experiment requires a 5-bit shared exponent.")
        if not 1 <= self.mantissa_bits <= 11:
            raise ValueError("mantissa_bits must be between 1 and 11.")
        if self.rounding != "nearest_even":
            raise ValueError("This experiment requires round-to-nearest-even.")

BFP = BFPConfig()
BFP.validate()
OUTPUT_DIR = Path("result")
ARCHIVE_PATH = Path("llama2-7b-bfp8-bfp4-g16-rne-no-lm-head-s2048.zip")

if not torch.cuda.is_available():
    raise RuntimeError("This notebook requires an NVIDIA CUDA GPU.")

torch.manual_seed(0)
torch.backends.cuda.matmul.allow_tf32 = False
print(f"Base config: {BFP}")
print(f"Sweep: {[f'BFP{1 + bits}' for bits in MANTISSA_BITS_SWEEP]}")

## BFP convention

Blocks contain 16 contiguous values along the Linear K dimension. Each weight row and token activation vector is partitioned independently. The shared exponent is the maximum unbiased exponent in the block: `shared_exp = max(floor(log2(abs(x))))`, clamped to signed E5. With M magnitude bits, `step = 2 ** (shared_exp - (M - 1))`; `torch.round(x / step)` performs round-to-nearest-even, followed by saturation to the symmetric M-bit magnitude range. BFP8 is 1S7M plus one shared E5 exponent per G16 block. This arithmetic fake-quantization path is equivalent to a bit-exact guard/sticky implementation for normal finite FP16 values under the same exponent, saturation, and subnormal contracts. Zero blocks use zero mantissas and canonical exponent zero.

In [ ]:
def _quantize_bfp_rows(rows, config, return_metadata=False):
    original_shape = rows.shape
    width = original_shape[-1]
    flat = rows.reshape(-1, width).float()
    padding = (-width) % config.block_size

    if padding:
        flat = F.pad(flat, (0, padding))

    padded_width = flat.size(1)
    blocks = flat.reshape(flat.size(0), -1, config.block_size)
    max_abs = blocks.abs().amax(dim=-1, keepdim=True)
    safe_max = max_abs.clamp_min(torch.finfo(torch.float32).tiny)
    raw_shared_exp = torch.floor(torch.log2(safe_max))

    exp_min = -(1 << (config.shared_exponent_bits - 1))
    exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
    zero_mask = max_abs == 0
    low_clamped = (~zero_mask) & (raw_shared_exp < exp_min)
    high_clamped = (~zero_mask) & (raw_shared_exp > exp_max)
    shared_exp = raw_shared_exp.clamp(exp_min, exp_max)
    shared_exp = torch.where(zero_mask, torch.zeros_like(shared_exp), shared_exp)

    step = torch.pow(2.0, shared_exp - (config.mantissa_bits - 1))
    mantissa_max = (1 << config.mantissa_bits) - 1
    mantissa = torch.round(blocks / step).clamp(-mantissa_max, mantissa_max)

    dequantized = (mantissa * step).reshape(flat.size(0), padded_width)
    dequantized = dequantized[:, :width].reshape(original_shape)
    dequantized = dequantized.to(rows.dtype)
    if not return_metadata:
        return dequantized
    return dequantized, {
        "shared_exp": shared_exp.squeeze(-1).to(torch.int16),
        "zero_mask": zero_mask.squeeze(-1),
        "low_clamped": low_clamped.squeeze(-1),
        "high_clamped": high_clamped.squeeze(-1),
    }


def quantize_bfp(tensor, config, chunk_rows, return_metadata=False):
    width = tensor.shape[-1]
    flat = tensor.reshape(-1, width)
    if flat.size(0) <= chunk_rows:
        return _quantize_bfp_rows(tensor, config, return_metadata)

    output = torch.empty_like(flat)
    metadata_parts = {
        "shared_exp": [],
        "zero_mask": [],
        "low_clamped": [],
        "high_clamped": [],
    } if return_metadata else None

    for start in range(0, flat.size(0), chunk_rows):
        end = min(start + chunk_rows, flat.size(0))
        if return_metadata:
            quantized, metadata = _quantize_bfp_rows(
                flat[start:end], config, return_metadata=True
            )
            output[start:end] = quantized
            for key in metadata_parts:
                metadata_parts[key].append(metadata[key])
        else:
            output[start:end] = _quantize_bfp_rows(flat[start:end], config)

    output = output.reshape_as(tensor)
    if not return_metadata:
        return output
    return output, {
        key: torch.cat(parts, dim=0) for key, parts in metadata_parts.items()
    }


@torch.no_grad()
def quantize_weight_in_place(weight, config):
    metadata_parts = {
        "shared_exp": [],
        "zero_mask": [],
        "low_clamped": [],
        "high_clamped": [],
    }
    for start in range(0, weight.size(0), config.weight_chunk_rows):
        end = min(start + config.weight_chunk_rows, weight.size(0))
        quantized, metadata = _quantize_bfp_rows(
            weight[start:end], config, return_metadata=True
        )
        weight[start:end].copy_(quantized)
        for key in metadata_parts:
            metadata_parts[key].append(metadata[key])
    return {
        key: torch.cat(parts, dim=0) for key, parts in metadata_parts.items()
    }


def _group_exponent_histogram(shared_exp, zero_mask, exp_min, num_bins):
    rows, groups = shared_exp.shape
    group_offsets = torch.arange(
        groups, device=shared_exp.device, dtype=torch.long
    ) * num_bins
    indices = shared_exp.long() - exp_min + group_offsets.unsqueeze(0)
    indices = indices[~zero_mask]
    return torch.bincount(
        indices.reshape(-1), minlength=groups * num_bins
    ).reshape(groups, num_bins)


def _histogram_record(counts, bins, zero_blocks, low_clamped, high_clamped):
    nonzero_bins = torch.nonzero(counts, as_tuple=False).flatten()
    minimum = None if nonzero_bins.numel() == 0 else int(bins[nonzero_bins[0]].item())
    maximum = None if nonzero_bins.numel() == 0 else int(bins[nonzero_bins[-1]].item())
    return {
        "bins": bins.detach().cpu().tolist(),
        "counts": counts.detach().cpu().tolist(),
        "nonzero_blocks": int(counts.sum().item()),
        "total_blocks": int(counts.sum().item() + zero_blocks.item()),
        "minimum": minimum,
        "maximum": maximum,
        "zero_blocks": int(zero_blocks.item()),
        "low_clamped_blocks": int(low_clamped.item()),
        "high_clamped_blocks": int(high_clamped.item()),
    }


class BFPLinear(nn.Module):
    def __init__(self, linear, config, weight_metadata):
        super().__init__()
        self.linear = linear
        self.config = config
        self.exp_min = -(1 << (config.shared_exponent_bits - 1))
        self.exp_max = (1 << (config.shared_exponent_bits - 1)) - 1
        self.num_exp_bins = self.exp_max - self.exp_min + 1
        weight_group_counts = _group_exponent_histogram(
            weight_metadata["shared_exp"], weight_metadata["zero_mask"],
            self.exp_min, self.num_exp_bins
        )
        self.register_buffer(
            "weight_group_counts_f64", weight_group_counts.to(torch.float64), persistent=False
        )
        self.register_buffer(
            "weight_shared_exp_counts", weight_group_counts.sum(dim=0), persistent=False
        )
        self.register_buffer(
            "activation_shared_exp_counts",
            torch.zeros(self.num_exp_bins, dtype=torch.int64, device=linear.weight.device),
            persistent=False,
        )
        self.register_buffer(
            "product_shared_exp_counts",
            torch.zeros(2 * self.num_exp_bins - 1, dtype=torch.int64, device=linear.weight.device),
            persistent=False,
        )
        self.register_buffer(
            "product_zero_partial_bases",
            torch.zeros((), dtype=torch.int64, device=linear.weight.device),
            persistent=False,
        )
        for prefix, metadata in (("weight", weight_metadata),):
            self.register_buffer(
                f"{prefix}_zero_blocks", metadata["zero_mask"].sum(dtype=torch.int64), persistent=False
            )
            self.register_buffer(
                f"{prefix}_low_clamped_blocks", metadata["low_clamped"].sum(dtype=torch.int64), persistent=False
            )
            self.register_buffer(
                f"{prefix}_high_clamped_blocks", metadata["high_clamped"].sum(dtype=torch.int64), persistent=False
            )
        for name in ("zero_blocks", "low_clamped_blocks", "high_clamped_blocks"):
            self.register_buffer(
                f"activation_{name}", torch.zeros((), dtype=torch.int64, device=linear.weight.device), persistent=False
            )

    @torch.no_grad()
    def _update_exponent_histograms(self, metadata):
        activation_group_counts = _group_exponent_histogram(
            metadata["shared_exp"], metadata["zero_mask"],
            self.exp_min, self.num_exp_bins
        )
        self.activation_shared_exp_counts.add_(activation_group_counts.sum(dim=0))
        self.activation_zero_blocks.add_(metadata["zero_mask"].sum(dtype=torch.int64))
        self.activation_low_clamped_blocks.add_(metadata["low_clamped"].sum(dtype=torch.int64))
        self.activation_high_clamped_blocks.add_(metadata["high_clamped"].sum(dtype=torch.int64))

        pair_counts = (
            activation_group_counts.transpose(0, 1).to(torch.float64)
            @ self.weight_group_counts_f64
        ).round().to(torch.int64)
        total_partial_bases = metadata["shared_exp"].numel() * self.linear.out_features
        self.product_zero_partial_bases.add_(total_partial_bases - pair_counts.sum())
        for activation_index in range(self.num_exp_bins):
            self.product_shared_exp_counts[
                activation_index : activation_index + self.num_exp_bins
            ].add_(pair_counts[activation_index])

    def export_exponent_histograms(self):
        shared_bins = torch.arange(
            self.exp_min, self.exp_max + 1, device=self.linear.weight.device
        )
        product_bins = torch.arange(
            2 * self.exp_min, 2 * self.exp_max + 1, device=self.linear.weight.device
        )
        weight = _histogram_record(
            self.weight_shared_exp_counts, shared_bins, self.weight_zero_blocks,
            self.weight_low_clamped_blocks, self.weight_high_clamped_blocks,
        )
        activation = _histogram_record(
            self.activation_shared_exp_counts, shared_bins, self.activation_zero_blocks,
            self.activation_low_clamped_blocks, self.activation_high_clamped_blocks,
        )
        product = {
            "definition": "weight_shared_exp + activation_shared_exp",
            "bins": product_bins.detach().cpu().tolist(),
            "counts": self.product_shared_exp_counts.detach().cpu().tolist(),
            "nonzero_partial_bases": int(self.product_shared_exp_counts.sum().item()),
            "zero_partial_bases": int(self.product_zero_partial_bases.item()),
        }
        product["total_partial_bases"] = (
            product["nonzero_partial_bases"] + product["zero_partial_bases"]
        )
        expected_product_total = (
            activation["total_blocks"] * self.linear.out_features
        )
        if product["total_partial_bases"] != expected_product_total:
            raise RuntimeError("Product shared-exponent histogram closure failed.")
        return {
            "input_features": self.linear.in_features,
            "output_features": self.linear.out_features,
            "groups_per_row": self.weight_group_counts_f64.size(0),
            "weight_shared_exponent": weight,
            "activation_shared_exponent": activation,
            "product_shared_exponent": product,
        }

    def forward(self, x):
        x_bfp, metadata = quantize_bfp(
            x, self.config, self.config.activation_chunk_rows, return_metadata=True
        )
        self._update_exponent_histograms(metadata)
        return F.linear(x_bfp, self.linear.weight, self.linear.bias).to(torch.float16)


def replace_linear_layers(module, config, prefix=""):
    replaced = {}

    for name, child in list(module.named_children()):
        full_name = f"{prefix}.{name}" if prefix else name

        if isinstance(child, nn.Linear):
            if full_name == "lm_head" and not config.quantize_lm_head:
                continue
            weight_metadata = quantize_weight_in_place(child.weight, config)
            wrapper = BFPLinear(child, config, weight_metadata)
            setattr(module, name, wrapper)
            replaced[full_name] = wrapper
        else:
            replaced.update(replace_linear_layers(child, config, full_name))

    return replaced


def export_layer_exponent_histograms(layers):
    records = []
    for layer_name, layer in layers.items():
        record = {"layer_name": layer_name, **layer.export_exponent_histograms()}
        records.append(record)
    return records


sample = torch.tensor([[1.0078125, 1.0234375, -1.0234375, 1.5]], device="cuda", dtype=torch.float16)
sample_q = _quantize_bfp_rows(sample, BFP)
assert sample_q.shape == sample.shape
assert sample_q.dtype == torch.float16
assert torch.isfinite(sample_q).all()
expected_q = torch.tensor([[1.0, 1.03125, -1.03125, 1.5]], device="cuda", dtype=torch.float16)
assert torch.equal(sample_q, expected_q), (sample, sample_q, expected_q)

## Load tokenizer and prepare the BFP sweep

Accept the LLaMA-2 license and add a Colab secret named `HF_TOKEN`. The tokenizer is loaded once. Each sweep iteration reloads the original FP16 model from the Hugging Face cache, then quantizes weights in place; activations are quantized on every Linear forward pass.

In [ ]:
token = os.getenv("HF_TOKEN")
if not token:
    try:
        from google.colab import userdata
        token = userdata.get("HF_TOKEN")
    except Exception:
        token = getpass("HF_TOKEN: ")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token)
print(f"Formats: {[f'BFP{1 + bits}' for bits in MANTISSA_BITS_SWEEP]}")

In [ ]:
dataset = load_dataset(DATASET_ID, DATASET_CONFIG, split=SPLIT)
text = "\n\n".join(dataset["text"])
input_ids = tokenizer(text, return_tensors="pt").input_ids
assert input_ids.numel() == BASELINE_TOKEN_COUNTS["source_input_tokens"]
print(f"WikiText-2 {SPLIT} tokens: {input_ids.numel():,}")

In [ ]:
@torch.inference_mode()
def evaluate_perplexity(model, input_ids, context_length, stride, drop_remainder):
    if stride != context_length:
        raise ValueError("Non-overlapping evaluation requires stride == context_length.")
    if not drop_remainder:
        raise ValueError("Paper-compatible evaluation requires drop_remainder=True.")
    if context_length > model.config.max_position_embeddings:
        raise ValueError("context_length exceeds the model context window.")

    device = next(model.parameters()).device
    sequence_length = input_ids.size(1)
    usable_length = sequence_length // context_length * context_length
    dropped_tokens = sequence_length - usable_length
    if usable_length == 0:
        raise ValueError("Input does not contain a complete context block.")

    total_nll = 0.0
    total_loss_tokens = 0
    total_blocks = usable_length // context_length

    torch.cuda.reset_peak_memory_stats(device)
    torch.cuda.synchronize(device)
    start_time = time.perf_counter()

    for begin in tqdm(range(0, usable_length, stride), total=total_blocks, desc="Evaluating"):
        end = begin + context_length
        batch = input_ids[:, begin:end].to(device)
        labels = batch.clone()

        loss = model(batch, labels=labels, use_cache=False).loss
        loss_tokens = labels[:, 1:].numel()
        total_nll += loss.float().item() * loss_tokens
        total_loss_tokens += loss_tokens

    torch.cuda.synchronize(device)
    elapsed_seconds = time.perf_counter() - start_time
    mean_nll = total_nll / total_loss_tokens

    return {
        "mean_nll": mean_nll,
        "perplexity": float(torch.exp(torch.tensor(mean_nll))),
        "source_input_tokens": sequence_length,
        "used_input_tokens": usable_length,
        "dropped_input_tokens": dropped_tokens,
        "evaluated_blocks": total_blocks,
        "evaluated_tokens": total_loss_tokens,
        "elapsed_seconds": elapsed_seconds,
        "tokens_per_second": total_loss_tokens / elapsed_seconds,
        "peak_gpu_memory_gib": torch.cuda.max_memory_allocated(device) / 2**30,
    }

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results = []
output_paths = []

for mantissa_bits in MANTISSA_BITS_SWEEP:
    config = replace(BFP, mantissa_bits=mantissa_bits)
    config.validate()
    bfp_bits = 1 + config.mantissa_bits
    output_path = OUTPUT_DIR / f"bfp{bfp_bits}-g{config.block_size}-rne-no-lm-head-s2048.json"

    print(f"\n{'=' * 72}")
    print(f"Running BFP{bfp_bits}: {config}")
    print('=' * 72)

    torch.manual_seed(0)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        dtype=torch.float16,
        device_map=0,
        attn_implementation="eager",
        token=token,
    )
    model.eval()
    model.config.use_cache = False

    quantized_layers = replace_linear_layers(model, config)
    quantized_layer_count = len(quantized_layers)
    torch.cuda.empty_cache()

    parameter_dtypes = {p.dtype for p in model.parameters() if p.is_floating_point()}
    parameter_devices = {p.device.type for p in model.parameters()}
    assert parameter_dtypes == {torch.float16}, parameter_dtypes
    assert parameter_devices == {"cuda"}, parameter_devices
    assert config.block_size == 16
    assert config.quantize_lm_head is False
    assert quantized_layer_count == 224, quantized_layer_count
    assert isinstance(model.lm_head, nn.Linear)

    print(f"Quantized Linear layers: {quantized_layer_count}")
    metrics = evaluate_perplexity(model, input_ids, CONTEXT_LENGTH, STRIDE, DROP_REMAINDER)
    for name, expected in BASELINE_TOKEN_COUNTS.items():
        if metrics[name] != expected:
            raise RuntimeError(
                f"Evaluation closure failed for {name}: "
                f"expected {expected}, got {metrics[name]}"
            )
    layer_exponent_histograms = export_layer_exponent_histograms(quantized_layers)
    if len(layer_exponent_histograms) != quantized_layer_count:
        raise RuntimeError("Layer exponent metadata count mismatch.")
    result = {
        "model": MODEL_ID,
        "model_revision": getattr(model.config, "_commit_hash", None),
        "tokenizer_class": tokenizer.__class__.__name__,
        "dataset": f"{DATASET_ID}/{DATASET_CONFIG}",
        "dataset_fingerprint": getattr(dataset, "_fingerprint", None),
        "split": SPLIT,
        "quantization": "decoder Linear W/A BFP fake quantization; FP16 lm_head",
        "format": f"BFP{bfp_bits} (1S{config.mantissa_bits}M + shared E{config.shared_exponent_bits})",
        "bfp_config": asdict(config),
        "sweep_mantissa_bits": list(MANTISSA_BITS_SWEEP),
        "shared_exponent_contract": {
            "selection": "maximum unbiased exponent in each contiguous G16 block",
            "formula": "max(floor(log2(abs(x)))) over nonzero block values",
            "semantics": "MSFP maximum exponent, not mantissa-LSB scale exponent",
            "encoding": "signed integer",
            "zero_block_exponent": 0,
        },
        "mantissa_rounding_contract": "round-to-nearest-even (torch.round), followed by symmetric saturation",
        "quantized_scope": {
            "operations": "LLaMA decoder nn.Linear modules",
            "weights": True,
            "activations": True,
            "lm_head": False,
            "attention_internal_matmul": False,
            "softmax": False,
            "layernorm": False,
        },
        "linear_output_dtype": "float16",
        "matmul_backend": "torch.nn.functional.linear with dequantized FP16 operands",
        "quantized_linear_layers": quantized_layer_count,
        "exponent_telemetry": {
            "storage": "per-layer exact histograms; raw exponent streams are not stored",
            "product_shared_exponent_definition": "weight_shared_exp + activation_shared_exp",
            "actual_partial_sum_leading_exponent_profiled": False,
            "layers": layer_exponent_histograms,
        },
        "attention_implementation": "eager",
        "context_length": CONTEXT_LENGTH,
        "stride": STRIDE,
        "evaluation_protocol": EVALUATION_PROTOCOL,
        "drop_remainder": DROP_REMAINDER,
        "validation": {
            "expected_quantized_linear_layers": 224,
            "expected_evaluation_counts": BASELINE_TOKEN_COUNTS,
            "evaluation_closure_passed": True,
            "histogram_closure_passed": True,
        },
        "baseline_perplexity": BASELINE_PPL,
        "delta_perplexity": None if BASELINE_PPL is None else metrics["perplexity"] - BASELINE_PPL,
        "gpu": torch.cuda.get_device_name(0),
        "cuda": torch.version.cuda,
        "python": platform.python_version(),
        "pytorch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        **metrics,
    }

    output_path.write_text(json.dumps(result, indent=2, ensure_ascii=False), encoding="utf-8")
    results.append(result)
    output_paths.append(output_path)
    print(json.dumps(result, indent=2, ensure_ascii=False))
    print(f"Saved: {output_path.resolve()}")

    del model, quantized_layers
    gc.collect()
    torch.cuda.empty_cache()

print("\nSweep complete:")
for result in results:
    print(f"{result['format'].split()[0]}: PPL={result['perplexity']:.6f}")

In [ ]:
with zipfile.ZipFile(ARCHIVE_PATH, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in output_paths:
        archive.write(path, arcname=path.name)

print(f"Created: {ARCHIVE_PATH.resolve()}")
from google.colab import files
files.download(str(ARCHIVE_PATH))